# Writing a layer, and letting fit() drive it

A Dense layer written as a keras.Layer subclass — build(), call(), and the lazy weight creation that makes input shapes optional.

**Runs on:** CPU — about 1 minute &nbsp;·&nbsp; **Slides:** [Chapter 3 — Introduction to TensorFlow, PyTorch, JAX, and Keras](../../../course-web-slides/ch03/index.html) &nbsp;·&nbsp; **Section:** 04 — Layers, models, and the training loop

---

## The layer

In [ ]:
import keras
from keras import ops

class SimpleDense(keras.Layer):
    def __init__(self, units, activation=None):
        super().__init__()
        self.units = units
        self.activation = activation

    def build(self, input_shape):
        # Called on the first invocation, once the input shape is known.
        # This is why you never have to declare input sizes in Keras.
        input_dim = input_shape[-1]
        self.W = self.add_weight(
            shape=(input_dim, self.units),
            initializer="glorot_uniform",
            name="kernel",
        )
        self.b = self.add_weight(
            shape=(self.units,), initializer="zeros", name="bias",
        )

    def call(self, inputs):
        y = ops.matmul(inputs, self.W) + self.b
        if self.activation is not None:
            y = self.activation(y)
        return y

## Weights appear on first call, not on construction

In [ ]:
layer = SimpleDense(units=32, activation=ops.relu)
print("weights before any call:", len(layer.weights))

import numpy as np
out = layer(np.random.random((2, 784)).astype("float32"))
print("weights after one call: ", len(layer.weights))
print("output shape:           ", out.shape)
print("kernel shape:           ", layer.W.shape)

Expected output:

```
weights before any call: 0
weights after one call:  2
output shape:            (2, 32)
kernel shape:            (784, 32)
```

This is *lazy building*, and it is why a Keras model can be written without ever stating an input size. The shape arrives with the first batch.

## Composing them

In [ ]:
from keras.datasets import mnist

model = keras.Sequential([
    SimpleDense(512, activation=ops.relu),
    SimpleDense(10, activation=ops.softmax),
])

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255
x_test = x_test.reshape(-1, 784).astype("float32") / 255

model.compile(optimizer="rmsprop",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(x_train, y_train, epochs=3, batch_size=128, verbose=2)
print("test:", model.evaluate(x_test, y_test, verbose=0))

A layer you wrote, driven by `fit()` you did not. Everything Keras offers — callbacks, metrics, validation splits, the progress bar — works because `SimpleDense` implements the two methods the framework asks for.

## What compile() actually stores

In [ ]:
print("optimizer:", model.optimizer.__class__.__name__)
print("loss:     ", model.loss)
print("metrics:  ", [m.name for m in model.metrics])
print("trainable weights:", len(model.trainable_weights))
print("total params:", model.count_params())

> **Note** — `compile()` does not compute anything. It records three decisions — how to measure wrongness, how to move the weights, and what else to report — and `fit()` reads them back.

---

## What to take away

- A layer needs `build()` and `call()`; everything else is inherited.
- Weights are created lazily on first call, which is why input shapes are optional.
- `compile()` records three decisions; `fit()` executes them.
- Custom layers get the whole framework for free — callbacks, metrics, validation.